# FlashEats — Class 5 Investigation
**Mission:** assemble trustworthy evidence about late deliveries.

Rules: no ML for first 90 minutes; preserve raw API responses; state your definition of late; never silently drop failures.

In [1]:
import os, zipfile, json, sqlite3, time, subprocess, sys
from pathlib import Path
import pandas as pd, requests

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE = Path('/content/flasheats_classroom')
if not BASE.exists() or not (BASE / 'database' / 'flasheats.db').exists():
    from google.colab import files
    print('Upload FlashEats_Classroom_Pack.zip')
    uploaded=files.upload(); zip_name=next(n for n in uploaded if n.endswith('.zip'))
    with zipfile.ZipFile(zip_name) as z: z.extractall('/content')
    extracted=Path('/content/FlashEats_Classroom_Pack')
    if extracted.exists(): extracted.rename(BASE)
print(BASE, BASE.exists())

D:\Code\JabTakNaaMaaneTinkuJiya\flasheats-classroom-pack True


## Challenge 1 — How large is the late-delivery problem?
Write down your definition of late, denominator, cancellation policy, and missing-timestamp policy before querying.

In [2]:
con=sqlite3.connect(BASE/'database'/'flasheats.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'",con)

,name
0,customers
1,drivers
2,restaurants
3,orders


In [3]:
query='''
-- YOUR SQL HERE
SELECT * FROM orders LIMIT 5;
'''
pd.read_sql(query,con)

,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,NaN,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


### Checkpoint
Compare counts with another team. If they differ, investigate row grain, duplicates, cancellations, missing timestamps, and definitions.

## Challenge 2 — Operations says traffic is the cause
Test the claim. Treat traffic/weather/distance as descriptive signals, not causal proof.

In [4]:
# Your analysis here

## Challenge 3 — Customer Support disagrees
Inspect support tickets. What are customers actually complaining about? Are ticket records unique?

In [5]:
tickets=pd.read_csv(BASE/'data'/'support_tickets.csv'); tickets.head()

,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still n...
2,T00003,NaN,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant say...
3,T00004,O00037,2026-08-09T21:09:00,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


## Challenge 4 — Retrieve Dispatch data reliably

In [6]:
!pip -q install flask
api_proc=subprocess.Popen([sys.executable,str(BASE/'api'/'mock_dispatch_api.py')],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
time.sleep(2)
requests.get('http://127.0.0.1:8000/health').json()

{'service': 'flasheats-dispatch-api', 'status': 'ok'}

In [7]:
url='http://127.0.0.1:8000/dispatch/orders'
r=requests.get(url,params={'page':1,'page_size':50},timeout=10)
print(r.status_code); payload=r.json(); print(payload.keys(), len(payload.get('data',[])), payload.get('has_more'))

200
dict_keys(['data', 'has_more', 'page', 'page_size', 'total_records']) 50 True


### Your task
Implement pagination + retry + raw-page preservation. Refuse to claim success if ingestion is incomplete.

In [8]:
RAW_DIR=BASE/'student_output'/'raw_dispatch'; RAW_DIR.mkdir(parents=True,exist_ok=True)
def fetch_all_dispatch_orders():
    records=[]
    page = 1
    while True:
        url = 'http://127.0.0.1:8000/dispatch/orders'
        retries = 3
        success = False
        while retries > 0 and not success:
            try:
                r = requests.get(url, params={'page': page, 'page_size': 50}, timeout=10)
                if r.status_code == 200:
                    payload = r.json()
                    with open(RAW_DIR / f'page_{page}.json', 'w') as out_f:
                        json.dump(payload, out_f)
                    records.extend(payload.get('data', []))
                    if not payload.get('has_more', False):
                        return records
                    page += 1
                    success = True
                else:
                    retries -= 1
                    time.sleep(1)
            except Exception:
                retries -= 1
                time.sleep(1)
        if not success:
            raise Exception(f'Failed to fetch page {page} after retries. Incomplete ingestion.')
    return records

## Challenge 5 — Driver events
What is observed vs inferred? Can you find a reliable driver-arrival-at-restaurant event?

In [9]:
with open(BASE/'data'/'driver_events.json') as f: driver_events=json.load(f)
driver_events[0]

{'driver_id': 'D001',
 'events': [{'order_id': 'O00162',
   'type': 'assigned',
   'timestamp': '2026-08-26T17:53:25.977803'},
  {'order_id': 'O00162',
   'type': 'gps_ping',
   'timestamp': '2026-08-26T18:10:59.372420',
   'lat': 12.826339,
   'lon': 77.448282},
  {'order_id': 'O00162',
   'type': 'gps_ping',
   'timestamp': '2026-08-26T18:28:32.767037',
   'lat': 12.876013,
   'lon': 77.477707},
  {'order_id': 'O00162',
   'type': 'gps_ping',
   'timestamp': '2026-08-26T18:46:06.161653',
   'lat': 12.925686,
   'lon': 77.507133},
  {'order_id': 'O00162',
   'type': 'gps_ping',
   'timestamp': '2026-08-26T19:03:39.556270',
   'lat': 12.97536,
   'lon': 77.536558},
  {'order_id': 'O00162',
   'type': 'picked_up',
   'timestamp': '2026-08-26T18:27:32.131791'},
  {'order_id': 'O00162',
   'type': 'delivered',
   'timestamp': '2026-08-26T19:21:12.950887'},
  {'order_id': 'O00180',
   'type': 'assigned',
   'timestamp': '2026-08-08T18:05:08.439625'},
  {'order_id': 'O00180',
   'type': 'gp

## Final FDE recommendation
Prepare one slide: problem size, evidence, uncertainty, missing instrumentation, next step.

**Would you build the AI delay predictor now? Why or why not?**